# RAG Step 1: Document Loading and Chunking

**Learning objective:** Load sample text/PDF content and compare fixed-size, character, recursive, sentence, and token-based chunking strategies.

**Environment:** Jupyter Notebook or Google Colab. Run cells from top to bottom. Install commands are included where needed.

---

## Learning outcomes

- Load text from sample files.
- Compare multiple chunking approaches.
- Evaluate chunk counts and trade-offs.


## Why Chunking Matters

- Embedding models have a **maximum input length** (e.g. ~256-512 tokens for many free models).
- A whole book or long article won't fit, and even if it did, one giant embedding would blur together
  too many different ideas — making search results useless.
- **Chunking** splits a document into smaller pieces so each chunk represents one focused idea,
  which produces much more accurate embeddings and search results later on.


### Setup — Install Required Libraries


In [ ]:
!pip install -q langchain langchain-text-splitters tiktoken pypdf fpdf2


## 1. Create a Sample Document (so this notebook runs with zero external downloads)

In real projects your document would come from a file, website, or database. Here we generate one
locally so the notebook is 100% reliable and network-independent.


In [ ]:
import os

os.makedirs("docs", exist_ok=True)

article_text = """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own. This approach powers everything from spam filters
to recommendation systems.

Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understanding the meaning of a sentence.

Natural Language Processing, or NLP, is the branch of AI that focuses specifically on human
language. NLP powers chatbots, translation tools, and search engines. A key building block of
modern NLP is the embedding, which converts words or sentences into numeric vectors that capture
meaning.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.

Vector Databases store embeddings and allow fast similarity search across millions of documents.
They are a core infrastructure piece behind modern RAG systems, chatbots, and recommendation
engines used by companies around the world today.
"""

with open("docs/sample_article.txt", "w") as f:
    f.write(article_text)

print("Document created: docs/sample_article.txt")
print("Total characters:", len(article_text))
print("Total words:", len(article_text.split()))


## 2. Loading a `.txt` Document

The simplest form of document loading — read the raw text from disk.


In [ ]:
with open("docs/sample_article.txt", "r") as f:
    loaded_text = f.read()

print("Loaded document preview:\n")
print(loaded_text[:300], "...")


## 3. Loading a `.pdf` Document

Most real-world knowledge bases include PDFs. We'll first generate a simple sample PDF (so this
notebook needs no external file), then load and extract its text using `pypdf`.


In [ ]:
from fpdf import FPDF

pdf = FPDF()
pdf.add_page()
pdf.set_font("Times", size=12)

for line in article_text.split("\n"):
    # Using an explicit width (e.g., 190mm for A4 page with margins) instead of 0
    pdf.multi_cell(190, 8, line)

pdf.output("docs/sample_article.pdf")
print("PDF created: docs/sample_article.pdf")


In [ ]:
from pypdf import PdfReader

reader = PdfReader("docs/sample_article.pdf")

pdf_text = ""
for page in reader.pages:
    pdf_text += page.extract_text() + "\n"

print("Pages found:", len(reader.pages))
print("\nExtracted PDF text preview:\n")
print(pdf_text[:300], "...")


**Discussion Point:** Loading is format-specific (`.txt`, `.pdf`, `.docx`, HTML, etc.) but the goal isalways the same — end up with **plain text** that we can chunk. From here on we'll work with the textwe loaded from the `.txt` file.


## 4. Chunking Strategy 1 — Fixed-Size Character Chunking (From Scratch)
The simplest strategy: cut the text every N characters, with some overlap between chunks so wedon't lose context at the boundaries.


In [ ]:
def fixed_size_chunking(text, chunk_size=300, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

fixed_chunks = fixed_size_chunking(loaded_text, chunk_size=300, overlap=50)

print("Number of chunks:", len(fixed_chunks))
print("\n--- Chunk 1 ---\n", fixed_chunks[0])
print("\n--- Chunk 2 ---\n", fixed_chunks[1])


**Discussion Point:** Notice fixed-size chunking can cut a sentence or word right in half — it doesn'tknow anything about paragraph or sentence boundaries. It's simple and fast, but not always ideal.


## 5. Chunking Strategy 2 — `CharacterTextSplitter` (LangChain)
LangChain provides ready-made splitters. `CharacterTextSplitter` is the library version of what wejust built by hand — it splits on a chosen separator (like a blank line) and respects `chunk_size`.


In [ ]:
from langchain_text_splitters import CharacterTextSplitter

char_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=300,
    chunk_overlap=50
)

char_chunks = char_splitter.split_text(loaded_text)

print("Number of chunks:", len(char_chunks))
for i, chunk in enumerate(char_chunks):
    print(f"\n--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)


## 6. Chunking Strategy 3 — `RecursiveCharacterTextSplitter` (Recommended Default)
This is the most commonly used splitter in real RAG systems. It tries to split on paragraph breaksfirst, then sentences, then words — only falling back to a hard character cut as a last resort. Thiskeeps chunks much more semantically coherent.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

recursive_chunks = recursive_splitter.split_text(loaded_text)

print("Number of chunks:", len(recursive_chunks))
for i, chunk in enumerate(recursive_chunks):
    print(f"\n--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)


**Discussion Point:** Compare this output to Strategy 1. The recursive splitter tends to end chunks atnatural breaks (end of a paragraph or sentence) instead of mid-word — this is why it's the most widelyused default in production RAG pipelines.


## 7. Chunking Strategy 4 — Sentence-Based Chunking
Sometimes we want every chunk to contain whole sentences only — useful when precision matters morethan fixed size (e.g. legal or medical documents).


In [ ]:
import re

def sentence_chunking(text, sentences_per_chunk=2):
    # Simple sentence splitter using punctuation (no extra downloads needed)
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s for s in sentences if s]

    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = " ".join(sentences[i:i + sentences_per_chunk])
        chunks.append(chunk)
    return chunks

sentence_chunks = sentence_chunking(loaded_text, sentences_per_chunk=2)

print("Number of chunks:", len(sentence_chunks))
for i, chunk in enumerate(sentence_chunks[:4]):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)


## 8. Chunking Strategy 5 — Token-Based
 models and LLMs actually have limits measured in **tokens**, not characters (roughly1 token ≈ 4 characters in English). Token-based chunking guarantees we never exceed a model's realinput limit.


In [ ]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

def token_chunking(text, max_tokens=60, overlap_tokens=10):
    tokens = encoding.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunks.append(encoding.decode(chunk_tokens))
        start += max_tokens - overlap_tokens
    return chunks

token_chunks = token_chunking(loaded_text, max_tokens=60, overlap_tokens=10)

print("Number of chunks:", len(token_chunks))
for i, chunk in enumerate(token_chunks[:3]):
    token_count = len(encoding.encode(chunk))
    print(f"\n--- Chunk {i+1} ({token_count} tokens) ---")
    print(chunk)


## 9. Comparing All Strategies Side by Side


In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Strategy": [
        "Fixed-size (scratch)",
        "CharacterTextSplitter",
        "RecursiveCharacterTextSplitter",
        "Sentence-based",
        "Token-based"
    ],
    "Number of Chunks": [
        len(fixed_chunks),
        len(char_chunks),
        len(recursive_chunks),
        len(sentence_chunks),
        len(token_chunks)
    ],
    "Avg Chunk Length (chars)": [
        round(sum(len(c) for c in fixed_chunks) / len(fixed_chunks), 1),
        round(sum(len(c) for c in char_chunks) / len(char_chunks), 1),
        round(sum(len(c) for c in recursive_chunks) / len(recursive_chunks), 1),
        round(sum(len(c) for c in sentence_chunks) / len(sentence_chunks), 1),
        round(sum(len(c) for c in token_chunks) / len(token_chunks), 1)
    ]
})

comparison


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.bar(comparison["Strategy"], comparison["Number of Chunks"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Number of Chunks")
plt.title("Number of Chunks Produced by Each Strategy")
plt.tight_layout()
plt.show()


## Key Takeaways (Recap for Students)

| Strategy | Best For | Watch Out For |
|---|---|---|
| Fixed-size (scratch) | Learning the core idea | Can cut sentences/words in half |
| `CharacterTextSplitter` | Simple, separator-based splitting | Still fairly rigid |
| `RecursiveCharacterTextSplitter` | Most general-purpose RAG use cases | Slightly more setup, but worth it |
| Sentence-based | Precision-sensitive domains (legal, medical) | Chunk sizes vary a lot |
| Token-based | Respecting an embedding/LLM's exact token limit | Requires a tokenizer library |

**`chunk_overlap`** (used in most strategies above) keeps a bit of the previous chunk's text at the
start of the next chunk, so an idea that spans a chunk boundary isn't completely lost.

This notebook produced chunks — the **next step in RAG** is turning each chunk into an embedding
and storing it in a vector database (see the earlier notebooks in this series).

### Practice Exercises
1. Change `chunk_size` from 300 to 150 and re-run `RecursiveCharacterTextSplitter` — how does the
   number of chunks change?
2. Try `sentences_per_chunk=1` and `sentences_per_chunk=4` in the sentence-based strategy and compare.
3. Load your own `.txt` or `.pdf` file (upload it to Colab's file panel) and run all 5 strategies on it.


In [ ]:
# ==========================================
# PRACTICE 1
# Compare chunk sizes: 300 and 150
# ==========================================

from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------
# 1. Sample text
# ------------------------------------------

text = """
Generative Artificial Intelligence is a technology that can create new
content such as text, images, audio, and computer code. Modern generative
AI systems are trained using large datasets and can generate useful content
based on instructions provided by users.

Large Language Models, also known as LLMs, are a major technology behind
many generative AI applications. These models process natural language and
can perform tasks such as answering questions, summarizing documents,
writing content, translating languages, and generating code.

Prompt engineering is the process of designing effective instructions for
an artificial intelligence model. A well-designed prompt can provide
context, specify the required output, and improve the quality of the
generated response.

Vector databases are commonly used in modern AI applications. They store
numerical representations of information called embeddings. These
embeddings allow systems to find information based on semantic similarity.

Retrieval Augmented Generation, or RAG, combines document retrieval with
a language model. Documents are divided into smaller chunks, converted
into embeddings, and stored in a vector database. When a user submits a
question, relevant chunks are retrieved and supplied to the language model
to generate a context-aware answer.
"""


# ------------------------------------------
# 2. Create chunks with size 300
# ------------------------------------------

splitter_large = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks_large = splitter_large.split_text(text)


# ------------------------------------------
# 3. Create chunks with size 150
# ------------------------------------------

splitter_small = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=50
)

chunks_small = splitter_small.split_text(text)


# ------------------------------------------
# 4. Build comparison table
# ------------------------------------------

comparison = pd.DataFrame({
    "Chunk Size": [300, 150],
    "Number of Chunks": [
        len(chunks_large),
        len(chunks_small)
    ]
})

print("CHUNK SIZE COMPARISON")
print("=====================")
print(comparison)


# ------------------------------------------
# 5. Display chunk counts
# ------------------------------------------

print("\nChunks with size 300:",
      len(chunks_large))

print("Chunks with size 150:",
      len(chunks_small))


# ------------------------------------------
# 6. Visualize comparison
# ------------------------------------------

plt.figure(figsize=(7, 5))

plt.bar(
    comparison["Chunk Size"].astype(str),
    comparison["Number of Chunks"]
)

plt.xlabel("Chunk Size")
plt.ylabel("Number of Chunks")
plt.title("Comparison of Chunk Size and Chunk Count")

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.5
)

plt.tight_layout()
plt.show()


In [ ]:
# ==========================================
# PRACTICE 2
# Compare 1 sentence vs 4 sentences per chunk
# ==========================================

import re
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------
# 1. Sample text
# ------------------------------------------

text = """
Generative AI can create new content from user instructions.
Large Language Models are trained on large collections of text.
These models can answer questions and generate useful responses.
They are commonly used in many modern AI applications.
Prompt engineering focuses on creating effective instructions for AI models.
A good prompt can provide context and clearly describe the expected output.
Prompts can also specify the format and style of the response.
This can improve the quality of generated results.
Embeddings represent text as numerical vectors.
Similar pieces of text usually have similar vector representations.
Vector databases store these embeddings for efficient similarity search.
They are widely used in semantic search applications.
RAG combines retrieval with a generative language model.
Documents are divided into smaller chunks before they are stored.
Relevant chunks can be retrieved when a user submits a question.
The retrieved information provides additional context for the language model.
This approach can help a system answer questions using external documents.
Chunking is therefore an important part of many RAG applications.
"""
# ------------------------------------------
# 2. Convert text into sentences
# ------------------------------------------

sentences = re.split(
    r'(?<=[.!?])\s+',
    text.strip()
)


# ------------------------------------------
# 3. Sentence chunking function
# ------------------------------------------

def create_sentence_chunks(sentence_list, group_size):

    result = []

    for start in range(
        0,
        len(sentence_list),
        group_size
    ):

        selected_sentences = sentence_list[
            start:start + group_size
        ]

        result.append(
            " ".join(selected_sentences)
        )

    return result


# ------------------------------------------
# 4. Generate chunks
# ------------------------------------------

one_sentence_chunks = create_sentence_chunks(
    sentences,
    1
)

four_sentence_chunks = create_sentence_chunks(
    sentences,
    4
)


# ------------------------------------------
# 5. Create comparison table
# ------------------------------------------

comparison = pd.DataFrame({

    "Sentences in Each Chunk": [
        1,
        4
    ],

    "Total Chunks": [
        len(one_sentence_chunks),
        len(four_sentence_chunks)
    ]
})


print("SENTENCE CHUNKING ANALYSIS")
print("==========================")

print(comparison)


# ------------------------------------------
# 6. Show sample chunks
# ------------------------------------------

print("\n--- 1 Sentence per Chunk ---")

for number, chunk in enumerate(
    one_sentence_chunks[:3],
    start=1
):
    print(f"\nChunk {number}:")
    print(chunk)


print("\n--- 4 Sentences per Chunk ---")

for number, chunk in enumerate(
    four_sentence_chunks[:2],
    start=1
):
    print(f"\nChunk {number}:")
    print(chunk)


# ------------------------------------------
# 7. Plot comparison
# ------------------------------------------

plt.figure(figsize=(7, 5))

plt.bar(
    comparison["Sentences in Each Chunk"].astype(str),
    comparison["Total Chunks"]
)

plt.xlabel("Sentences in Each Chunk")

plt.ylabel("Total Number of Chunks")

plt.title(
    "Comparison of Sentence Grouping"
)

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.5
)

plt.tight_layout()

plt.show()


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter
)

# ==========================================
# Sample document
# ==========================================

text = """
Generative AI can create text, images, code, and other types of content.
Large Language Models are trained using very large collections of data.
These models can understand questions and generate natural language responses.
Prompt engineering helps users provide better instructions to AI models.
Embeddings convert text into numerical representations of its meaning.
Similar texts usually have similar embedding representations.
Vector databases are designed to store and search these embeddings.
Semantic search retrieves information based on meaning rather than exact words.
RAG combines document retrieval with a generative language model.
Before retrieval, large documents are usually divided into smaller chunks.
The retrieved chunks provide additional context to the language model.
Therefore, selecting an appropriate chunking strategy is important for RAG.
"""


# ==========================================
# 1. Fixed-length chunking
# ==========================================

fixed_chunks = [
    text[start:start + 250]
    for start in range(0, len(text), 250)
]


# ==========================================
# 2. Character-based chunking
# ==========================================

character_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=250,
    chunk_overlap=25
)

character_chunks = character_splitter.split_text(text)


# ==========================================
# 3. Recursive chunking
# ==========================================

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=25
)

recursive_chunks = recursive_splitter.split_text(text)


# ==========================================
# 4. Sentence-based chunking
# ==========================================

sentences = re.split(
    r'(?<=[.!?])\s+',
    text.strip()
)

sentence_chunks = [
    " ".join(sentences[index:index + 3])
    for index in range(0, len(sentences), 3)
]


# ==========================================
# 5. Word-based chunking
# ==========================================

words = text.split()

word_chunks = [
    " ".join(words[index:index + 40])
    for index in range(0, len(words), 40)
]


# ==========================================
# Combine all strategies
# ==========================================

chunk_sets = [
    fixed_chunks,
    character_chunks,
    recursive_chunks,
    sentence_chunks,
    word_chunks
]


# ==========================================
# Create comparison table
# ==========================================

comparison = pd.DataFrame({
    "Chunking Method": [
        "Fixed Length",
        "Character",
        "Recursive",
        "Sentence",
        "Word-based"
    ],

    "Chunk Count": [
        len(chunks)
        for chunks in chunk_sets
    ],

    "Average Length": [
        round(
            sum(len(chunk) for chunk in chunks) / len(chunks),
            1
        )
        for chunks in chunk_sets
    ]
})


# ==========================================
# Display results
# ==========================================

print("CHUNKING METHODS ANALYSIS")
print("=" * 45)

print(
    comparison.to_string(index=False)
)


# ==========================================
# Visual comparison
# ==========================================

plt.figure(figsize=(9, 5))

plt.bar(
    comparison["Chunking Method"],
    comparison["Chunk Count"]
)

plt.xlabel("Chunking Method")
plt.ylabel("Total Chunks")

plt.title(
    "Five Different Text Chunking Methods"
)

plt.xticks(rotation=20)

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.5
)

plt.tight_layout()

plt.show()


## Learning reflection

After running the notebook, write a few sentences in your own words:

1. What did you implement or test?
2. What result surprised you, and why?
3. What would you change or investigate next?
